In [7]:
#작업 준비
import nltk
from nltk.corpus import movie_reviews

from nltk.probability import ConditionalFreqDist
from nltk.probability import ConditionalProbDist
from nltk.probability import MLEProbDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wordcloud import WordCloud
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.layers import Dense,LSTM,BatchNormalization
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping,ModelCheckpoint
from keras.models import load_model

In [11]:
from nltk.util import ngrams
from nltk import word_tokenize
data_l=[]
for i in movie_reviews.sents():
    ng_d=ngrams(i,2,pad_left=True,pad_right=True,left_pad_symbol='SS',right_pad_symbol='SE')
    data_l+=[x for x in ng_d]
cfd=ConditionalFreqDist(data_l)

In [13]:
cfd.conditions()

['SS',
 'plot',
 ':',
 'two',
 'teen',
 'couples',
 'go',
 'to',
 'a',
 'church',
 'party',
 ',',
 'drink',
 'and',
 'then',
 'drive',
 '.',
 'they',
 'get',
 'into',
 'an',
 'accident',
 'one',
 'of',
 'the',
 'guys',
 'dies',
 'but',
 'his',
 'girlfriend',
 'continues',
 'see',
 'him',
 'in',
 'her',
 'life',
 'has',
 'nightmares',
 'what',
 "'",
 's',
 'deal',
 '?',
 'watch',
 'movie',
 '"',
 'sorta',
 'find',
 'out',
 'critique',
 'mind',
 '-',
 'fuck',
 'for',
 'generation',
 'that',
 'touches',
 'on',
 'very',
 'cool',
 'idea',
 'presents',
 'it',
 'bad',
 'package',
 'which',
 'is',
 'makes',
 'this',
 'review',
 'even',
 'harder',
 'write',
 'since',
 'i',
 'generally',
 'applaud',
 'films',
 'attempt',
 'break',
 'mold',
 'mess',
 'with',
 'your',
 'head',
 'such',
 '(',
 'lost',
 'highway',
 '&',
 'memento',
 ')',
 'there',
 'are',
 'good',
 'ways',
 'making',
 'all',
 'types',
 'these',
 'folks',
 'just',
 'didn',
 't',
 'snag',
 'correctly',
 'seem',
 'have',
 'taken',
 'pr

In [15]:
cpd=ConditionalProbDist(cfd,MLEProbDist)

In [21]:
cpd['the'].prob('movie')

0.0280547243528597

In [23]:
cpd['movie'].prob('.')

0.13897071564720154

In [25]:
cpd['.'].prob('movie')

0.0

In [167]:
import random
random.seed(10)
cpd['SS'].generate()

'she'

In [169]:
cpd['she'].generate()

'and'

In [171]:
t='SS'
random.seed(20)
while True:
    t=cpd[t].generate()
    if t=='SE':
        break
    print(t,end=' ')   

jacob ( deborah van damme bums will call believing that " paradise is the dread and you can reach space . 

In [175]:
'가' in cpd

False

In [199]:
def 문장생성(c='SS',s_n=10):
    random.seed(s_n)
    add_data_l=[]
    while True:
        if c not in cpd:
            break
        w = cpd[c].generate()
        if w=='SE':
            break
        elif w in ['i','ii','iii']:
            w2=w.upper()
        else:
            w2=w

        if c=='SS':
            add_data_l.append(w2.title())
        elif c in ["'",'"','(']:
            add_data_l.append(w2)
        elif w in ["'",'"',".",',',':',"?",';']:
            add_data_l.append(w2)
        else:
            add_data_l.append(' '+w2)

        c=w
    if len(add_data_l)>0:
        return ''.join(add_data_l)
    return "문장을 생성 할 수 없습니다."

            

In [201]:
문장생성()

'She and fine effect; frankly, and doing his son becomes shockingly lazy shortcut to her mother of"story"'

In [203]:
문장생성(s_n=20)

'Jacob (deborah van damme bums will call believing that"paradise is the dread and you can reach space.'

In [224]:
data=pd.read_table('data2.txt')[['document',	'label']]
#[for 문장_data in data ]


In [228]:
X=data[data.label==0].document
X2=data[data.label==1].document

In [230]:
X

0                                       아 더빙.. 진짜 짜증나네요 목소리
2                                         너무재밓었다그래서보는것을추천한다
3                             교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
5             막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.
6                                     원작의 긴장감을 제대로 살려내지못했다.
                                ...                        
149990                                         이걸 영화라고 찎었냐?
149992    공포나 재난영화가 아니라 아예 대놓고 비급 크리쳐개그물임ㅋㅋ 음악 완전 흥겹다ㅋ 5...
149995                                  인간이 문제지.. 소는 뭔죄인가..
149997                      이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?
149999                             한국 영화 최초로 수간하는 내용이 담긴 영화
Name: document, Length: 75173, dtype: object

In [242]:
x_data=[i for i in X]
x_data

['아 더빙.. 진짜 짜증나네요 목소리',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.',
 '원작의 긴장감을 제대로 살려내지못했다.',
 '별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단 낫겟다 납치.감금만반복반복..이드라마는 가족도없다 연기못하는사람만모엿네',
 '울면서 손들고 횡단보도 건널때 뛰쳐나올뻔 이범수 연기 드럽게못해',
 '취향은 존중한다지만 진짜 내생에 극장에서 본 영화중 가장 노잼 노감동임 스토리도 어거지고 감동도 어거지',
 '굿바이 레닌 표절인것은 이해하는데 왜 뒤로 갈수록 재미없어지냐',
 '재미없다 지루하고. 같은 음식 영화인데도 바베트의 만찬하고 넘 차이남....바베트의 만찬은 이야기도 있고 음식 보는재미도 있는데 ; 이건 볼게없다 음식도 별로 안나오고, 핀란드 풍경이라도 구경할랫는데 그것도 별로 안나옴 ㅡㅡ',
 '주제는 좋은데 중반부터 지루하다',
 '다 짤랐을꺼야. 그래서 납득할 수 없었던거야.. 그럴꺼야.. 꼭 그랬던걸꺼야..',
 '카밀라벨 발연기',
 '졸쓰레기 진부하고말도안됌ㅋㅋ 아..시간아까워',
 '1%라도 기대했던 내가 죄인입니다 죄인입니다....',
 '키이라 나이틀리가 연기하고자 했던건 대체 정신장애일까 틱장애일까',
 '포스터는 있어보이는데 관객은 114명이네',
 "'다 알바생인가 내용도 없고 무서운거도 없고 웃긴거도 하나도 없음 완전 별싱거운 영화.ㅇ.ㅇ내ㅇ시간 넘 아까움 .. . 완전 낚임",
 '평점에속지마시길시간낭비 돈낭비임',
 '리얼리티가 뛰어나긴 한데 큰 공감은 안간다. 이민기캐릭터는 정신의학상 분노조절장애 초기 증상일거다. 툭하면 사람패고 욕하고 물건 파손하고.. 조금 오바였음. 극 초반엔 신선했는데 가면 갈수록 이민기 정신상태 공감불가.',
 '마이너스는 왜없냐 ㅋ 뮤비 보고 영화수준 딱 알만하더군 ㅉㅉ 북한에서 이런거

In [248]:
x_data[:10]

['아 더빙.. 진짜 짜증나네요 목소리',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.',
 '원작의 긴장감을 제대로 살려내지못했다.',
 '별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단 낫겟다 납치.감금만반복반복..이드라마는 가족도없다 연기못하는사람만모엿네',
 '울면서 손들고 횡단보도 건널때 뛰쳐나올뻔 이범수 연기 드럽게못해',
 '취향은 존중한다지만 진짜 내생에 극장에서 본 영화중 가장 노잼 노감동임 스토리도 어거지고 감동도 어거지',
 '굿바이 레닌 표절인것은 이해하는데 왜 뒤로 갈수록 재미없어지냐',
 '재미없다 지루하고. 같은 음식 영화인데도 바베트의 만찬하고 넘 차이남....바베트의 만찬은 이야기도 있고 음식 보는재미도 있는데 ; 이건 볼게없다 음식도 별로 안나오고, 핀란드 풍경이라도 구경할랫는데 그것도 별로 안나옴 ㅡㅡ']

In [258]:
from konlpy.tag import Okt
okt=Okt()
def tk_f(문장):
    tk=['/'.join(j) for j in okt.pos(i)]
    return tk
tk_f(x_data[0])    

['재미없다/Adjective',
 '지루하고/Adjective',
 './Punctuation',
 '같은/Adjective',
 '음식/Noun',
 '영화/Noun',
 '인데/Josa',
 '도/Noun',
 '바베트/Noun',
 '의/Josa',
 '만찬/Noun',
 '하고/Josa',
 '넘/Verb',
 '차이남/Verb',
 '..../Punctuation',
 '바베트/Noun',
 '의/Josa',
 '만찬/Noun',
 '은/Josa',
 '이야기/Noun',
 '도/Josa',
 '있고/Adjective',
 '음식/Noun',
 '보는/Verb',
 '재미/Noun',
 '도/Josa',
 '있는데/Adjective',
 ';/Punctuation',
 '이건/Noun',
 '볼/Noun',
 '게/Josa',
 '없다/Adjective',
 '음식/Noun',
 '도/Josa',
 '별로/Noun',
 '안/VerbPrefix',
 '나오고/Verb',
 ',/Punctuation',
 '핀란드/Noun',
 '풍경/Noun',
 '이라도/Josa',
 '구/Modifier',
 '경/Modifier',
 '할랫/Noun',
 '는데/Verb',
 '그것/Noun',
 '도/Josa',
 '별로/Noun',
 '안/VerbPrefix',
 '나옴/Verb',
 'ㅡㅡ/KoreanParticle']

In [260]:
from konlpy.tag import Okt
from tqdm import tqdm
okt=Okt()
end_d=[]
for 문장 in tqdm(x_data):
    ck=tk_f(문장)
    ng_d=ngrams(ck,2,pad_left=True,pad_right=True,left_pad_symbol='SS',right_pad_symbol='SE')
    end_d+=[t for t in ng_d]

100%|████████████████████████████████████████████████████████████████████████████| 75173/75173 [15:50<00:00, 79.05it/s]


In [262]:
len(end_d)

3908996

In [268]:
import pickle
pickle.dump(end_d,open('data.p','wb'))

In [270]:
end_d=pickle.load(open('data.p','rb'))

In [272]:
len(end_d)

3908996

In [274]:
end_d[:10]

[('SS', '재미없다/Adjective'),
 ('재미없다/Adjective', '지루하고/Adjective'),
 ('지루하고/Adjective', './Punctuation'),
 ('./Punctuation', '같은/Adjective'),
 ('같은/Adjective', '음식/Noun'),
 ('음식/Noun', '영화/Noun'),
 ('영화/Noun', '인데/Josa'),
 ('인데/Josa', '도/Noun'),
 ('도/Noun', '바베트/Noun'),
 ('바베트/Noun', '의/Josa')]

In [276]:
cfd=ConditionalFreqDist(end_d)

In [278]:
cpd=ConditionalProbDist(cfd,MLEProbDist)

In [284]:
st='SS'
all_l=[]
random.seed(8)
while True:
    st=cpd[st].generate()
    if st=='SE':
        break
    d=st.split('/')[0]
    all_l.append(d)
' '.join(all_l)

'재미없다 지루하고 . 같은 음식 영화 인데 도 바베트 의 만찬 하고 넘 차이남 .... 바베트 의 만찬 하고 넘 차이남 .... 바베트 의 만찬 은 이야기 도 있는데 ; 이건 볼 게 없다 음식 영화 인데 도 바베트 의 만찬 은 이야기 도 있는데 ; 이건 볼 게 없다 음식 도 별로 안 나오고 , 핀란드 풍경 이라도 구 경 할랫 는데 그것 도 있는데 ; 이건 볼 게 없다 음식 보는 재미 도 별로 안 나옴 ㅡㅡ'

In [286]:
def 문장생성(c='SS',s_n=10):
    random.seed(s_n)
    add_data_l=[]
    while True:
        if c not in cpd:
            break
        w = cpd[c].generate()
        if w=='SE':
            break
        w2=w.split('/')[0]
        pos=w.split('/')[1]

        if c=='SS':
            add_data_l.append(w2.title())
        elif c in ["'",'"','(']:
            add_data_l.append(w2)
        elif w in ["'",'"',".",',',':',"?",';']:
            add_data_l.append(w2)
        elif pos in ['Josa','Punctuation','Suffix']:
            add_data_l.append(w2)
        elif w in ['임/Noun','것/Noun','는걸/None','되다/Verb','이다/Verb','이다/Adjective']:
            add_data_l.append(w2)
        else:
            add_data_l.append(' '+w2)

        c=w
    if len(add_data_l)>0:
        return ''.join(add_data_l)
    return "문장을 생성 할 수 없습니다."

            

In [306]:
문장생성(s_n=2)

'재미없다 지루하고. 같은 음식도 별로 안 나옴 ㅡㅡ'

# Q1 한국어 데이터를 EDA 하고 데이터에 맞는 분류기를 만드시오

In [318]:
data=pd.read_table('steam.txt',names=['y','x'])

# Q2 한국어 데이터를 이용하여 문장생성기를 만드시오

In [321]:
data2=pd.read_table('steam.txt',names=['y','x'])

In [323]:
from keras.models import load_model
import pickle
전저리된_data=pickle.load(open('data.pickle','rb'))
load_model=load_model('best_model.keras')

In [325]:
전저리된_data.keys()

dict_keys(['학습_데이터', '학습_결과', '태스트_데이터', '태스트_결과', '검증_데이터', '검증_결과', '토큰', '단어수', '불용어', '입력길이'])

In [327]:
load_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 54, 128)             │       1,614,592 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 54, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 54, 128)             │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,634,693 (21.49 MB)

 Trainable params: 1,878,145 (7.16 MB)

 Non-trainable params: 256 (1.00 KB)

 Optimizer params: 3,756,292 (14.33 MB)

In [331]:
tr_x=전저리된_data['학습_데이터']
tr_y=전저리된_data['학습_결과']
tk=전저리된_data['토큰']
w_l=전저리된_data['입력길이']
w_n=전저리된_data['단어수']

In [337]:
tr_x.shape,w_l,w_n

((55570, 54), 54, 12614)

In [339]:
tk

In [345]:
load_model.layers[0].weights

[<Variable path=embedding/embeddings, shape=(12614, 128), dtype=float32, value=[[ 0.01707767 -0.01155467 -0.01993956 ... -0.00426042 -0.03900206
   -0.00083655]
  [ 0.02709012 -0.04348643  0.01857319 ... -0.02040106  0.04985164
   -0.01105188]
  [-0.01945842  0.092024    0.05823547 ... -0.00316623 -0.11072095
    0.01384255]
  ...
  [-0.00546863 -0.01123792  0.00416765 ...  0.0563563   0.06526702
    0.03845752]
  [-0.0505782  -0.03658757 -0.05648678 ...  0.04142939  0.01530125
   -0.00677049]
  [ 0.02649307 -0.01416116  0.03576266 ...  0.08457797  0.05452971
   -0.07509186]]>]

In [349]:
tk.word_counts

OrderedDict([('꼬맹이', 5),
             ('바드', 1),
             ('귀여운', 142),
             ('모험', 64),
             ('춤', 11),
             ('하고', 3457),
             ('노래', 171),
             ('독특하고', 17),
             ('딱', 469),
             ('그냥', 2427),
             ('저', 1123),
             ('냥', 179),
             ('할만', 790),
             ('함', 1831),
             ('영어', 548),
             ('안', 4537),
             ('어려움', 259),
             ('지금', 854),
             ('겜', 4482),
             ('해봤는데', 318),
             ('뭔', 464),
             ('게임', 25344),
             ('되질', 33),
             ('않는다', 196),
             ('서버', 1016),
             ('관리', 144),
             ('개', 2383),
             ('씹', 254),
             ('좃', 27),
             ('같이', 839),
             ('하네', 109),
             ('못', 3388),
             ('이딴', 185),
             ('걸', 1165),
             ('만원', 379),
             ('받고', 164),
             ('판다', 15),
             ('고', 2570),
             ('

In [353]:
d=tk.word_index.items()

In [355]:
pr_data=sorted(d,key=lambda x:x[1])
pr_data

[('게임', 1),
 ('에', 2),
 ('로', 3),
 ('한', 4),
 ('안', 5),
 ('겜', 6),
 ('너무', 7),
 ('다', 8),
 ('할', 9),
 ('시간', 10),
 ('플레이', 11),
 ('하고', 12),
 ('못', 13),
 ('적', 14),
 ('하는', 15),
 ('것', 16),
 ('좀', 17),
 ('에서', 18),
 ('때', 19),
 ('입니다', 20),
 ('나', 21),
 ('사람', 22),
 ('추천', 23),
 ('거', 24),
 ('재미', 25),
 ('고', 26),
 ('인', 27),
 ('그냥', 28),
 ('개', 29),
 ('스토리', 30),
 ('생각', 31),
 ('하면', 32),
 ('내', 33),
 ('더', 34),
 ('게', 35),
 ('왜', 36),
 ('수', 37),
 ('잘', 38),
 ('갓', 39),
 ('과', 40),
 ('진짜', 41),
 ('그', 42),
 ('이다', 43),
 ('정말', 44),
 ('보다', 45),
 ('함', 46),
 ('와', 47),
 ('아', 48),
 ('같은', 49),
 ('그래픽', 50),
 ('버그', 51),
 ('돈', 52),
 ('합니다', 53),
 ('정도', 54),
 ('요', 55),
 ('하', 56),
 ('해서', 57),
 ('뭐', 58),
 ('분', 59),
 ('환불', 60),
 ('중', 61),
 ('친구', 62),
 ('인데', 63),
 ('면', 64),
 ('느낌', 65),
 ('있는', 66),
 ('처음', 67),
 ('까지', 68),
 ('하지만', 69),
 ('일', 70),
 ('하나', 71),
 ('좋은', 72),
 ('말', 73),
 ('임', 74),
 ('지', 75),
 ('감', 76),
 ('멀티', 77),
 ('난이도', 78),
 ('걸', 79),
 ('실행', 80),
 ('했

In [359]:
len(tk.word_index)

59257

In [363]:
w_l_data=list(tk.word_index.keys())
w_l_data.insert(0,'OOV')

In [367]:
w_l_data=w_l_data[:12614]

In [369]:
w_l_data

['OOV',
 '게임',
 '에',
 '로',
 '한',
 '안',
 '겜',
 '너무',
 '다',
 '할',
 '시간',
 '플레이',
 '하고',
 '못',
 '적',
 '하는',
 '것',
 '좀',
 '에서',
 '때',
 '입니다',
 '나',
 '사람',
 '추천',
 '거',
 '재미',
 '고',
 '인',
 '그냥',
 '개',
 '스토리',
 '생각',
 '하면',
 '내',
 '더',
 '게',
 '왜',
 '수',
 '잘',
 '갓',
 '과',
 '진짜',
 '그',
 '이다',
 '정말',
 '보다',
 '함',
 '와',
 '아',
 '같은',
 '그래픽',
 '버그',
 '돈',
 '합니다',
 '정도',
 '요',
 '하',
 '해서',
 '뭐',
 '분',
 '환불',
 '중',
 '친구',
 '인데',
 '면',
 '느낌',
 '있는',
 '처음',
 '까지',
 '하지만',
 '일',
 '하나',
 '좋은',
 '말',
 '임',
 '지',
 '감',
 '멀티',
 '난이도',
 '걸',
 '실행',
 '했는데',
 '구매',
 '저',
 '많이',
 '해',
 '없는',
 '엔딩',
 '없다',
 '그리고',
 '랑',
 '서',
 '퍼즐',
 '계속',
 '때문',
 '없음',
 '서버',
 '무료',
 '다시',
 '세',
 '근데',
 '하지',
 '키',
 '성',
 '다른',
 '시작',
 '점',
 '이나',
 '이런',
 '자체',
 '해도',
 '도전',
 '별로',
 '원',
 '진행',
 '조작',
 '이라',
 '제',
 '한번',
 '라',
 '한글',
 '존나',
 '모드',
 '매우',
 '지금',
 '가격',
 '부터',
 '시발',
 '되는',
 '네',
 '같이',
 '시리즈',
 '노잼',
 '과제',
 '보면',
 '스팀',
 '없고',
 '캐릭터',
 '한다',
 '할만',
 '그래도',
 '하기',
 '조금',
 '패치',
 '한글화',
 '같다',
 '있다',
 '보고',
 '이건

In [371]:
ws=load_model.layers[0].get_weights()[0]

In [375]:
ws.shape

(12614, 128)

In [379]:
ws[1].shape

(128,)

In [385]:
import io
out_v=io.open('v_data.tsv','w',encoding='utf-8')
out_w=io.open('w_data.tsv','w',encoding='utf-8')
for i,w in enumerate(w_l_data):
    if i==0:
        continue
    vec=ws[i]
    out_v.write('\t'.join([str(i) for i in vec])+'\n')
    out_w.write(w+'\n')
out_v.close()
out_w.close()

# 텍스트 데이터
## 텍스트 데이터 전처리
- 코퍼스 기준
- 단어 도출
- 단어 차원화
    - 국소/분산
- 피처 도출
  
## 텍스트 데이터 활용
- 통계/머신/딥
- 언어 모델
    - 문장 생성(다중 클래스 분류)
    - 내용 분류
    - 내용 분석
    - 내용 요약
- 결론 도출

In [392]:
from keras.utils import pad_sequences,to_categorical
from keras import Sequential
from keras.layers import Embedding,SimpleRNN,GRU,LSTM,Dense

import numpy as np
import pandas as pd

In [398]:
ck_data=x_data[:5]
ck_data

['아 더빙.. 진짜 짜증나네요 목소리',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.',
 '원작의 긴장감을 제대로 살려내지못했다.']

In [410]:
''.join(ck_data).split()

['아',
 '더빙..',
 '진짜',
 '짜증나네요',
 '목소리너무재밓었다그래서보는것을추천한다교도소',
 '이야기구먼',
 '..솔직히',
 '재미는',
 '없다..평점',
 '조정막',
 '걸음마',
 '뗀',
 '3세부터',
 '초등학교',
 '1학년생인',
 '8살용영화.ㅋㅋㅋ...별반개도',
 '아까움.원작의',
 '긴장감을',
 '제대로',
 '살려내지못했다.']

In [416]:
sorted(set(''.join(ck_data).split()))

['..솔직히',
 '1학년생인',
 '3세부터',
 '8살용영화.ㅋㅋㅋ...별반개도',
 '걸음마',
 '긴장감을',
 '더빙..',
 '뗀',
 '목소리너무재밓었다그래서보는것을추천한다교도소',
 '살려내지못했다.',
 '아',
 '아까움.원작의',
 '없다..평점',
 '이야기구먼',
 '재미는',
 '제대로',
 '조정막',
 '진짜',
 '짜증나네요',
 '초등학교']

In [418]:
tk_idx={w:i+1 for i,w in enumerate(sorted(set(''.join(ck_data).split())))}
idx_tk={i:w for w,i in tk_idx.items()}

In [426]:
vocab_size=len(tk_idx)+1

In [446]:
#글자간의 연결성을 이용하여 단어 도출
#문장 정립

In [448]:
ck_data

['아 더빙.. 진짜 짜증나네요 목소리',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.',
 '원작의 긴장감을 제대로 살려내지못했다.']

In [482]:
okt=Okt()
def tk_f(문장):
    return [f'{w}/{pos}' for w,pos in okt.pos(문장,stem=True)]
tk_ck_data=[tk_f(s) for s in ck_data]
tk_ck_data

[['아/Exclamation',
  '더빙/Noun',
  '../Punctuation',
  '진짜/Noun',
  '짜증나다/Adjective',
  '목소리/Noun'],
 ['너/Modifier',
  '무재/Noun',
  '밓었/Noun',
  '다그/Noun',
  '래서/Noun',
  '보다/Verb',
  '추천/Noun',
  '한/Josa',
  '다/Adverb'],
 ['교도소/Noun',
  '이야기/Noun',
  '구먼/Noun',
  '../Punctuation',
  '솔직하다/Adjective',
  '재미/Noun',
  '는/Josa',
  '없다/Adjective',
  '../Punctuation',
  '평점/Noun',
  '조정/Noun'],
 ['막/Noun',
  '걸음/Noun',
  '마/Noun',
  '떼다/Verb',
  '3/Number',
  '세/Noun',
  '부터/Josa',
  '초등학교/Noun',
  '1/Number',
  '학년/Noun',
  '생인/Noun',
  '8/Number',
  '살다/Verb',
  '영화/Noun',
  './Punctuation',
  'ㅋㅋㅋ/KoreanParticle',
  '.../Punctuation',
  '별/Modifier',
  '반개/Noun',
  '도/Josa',
  '아깝다/Adjective',
  '움/Noun',
  './Punctuation'],
 ['원작/Noun',
  '의/Josa',
  '긴장감/Noun',
  '을/Josa',
  '제대로/Noun',
  '살리다/Verb',
  '하다/Verb',
  './Punctuation']]

In [484]:
tk_ck=sorted(set([단어 for 문장 in tk_ck_data for 단어 in 문장]))
tk_idx={w:i+1 for i,w in enumerate(tk_ck)}
idx_tk={i:w for w,i in tk_idx.items()}

In [486]:
tk_idx

{'.../Punctuation': 1,
 '../Punctuation': 2,
 './Punctuation': 3,
 '1/Number': 4,
 '3/Number': 5,
 '8/Number': 6,
 'ㅋㅋㅋ/KoreanParticle': 7,
 '걸음/Noun': 8,
 '교도소/Noun': 9,
 '구먼/Noun': 10,
 '긴장감/Noun': 11,
 '너/Modifier': 12,
 '는/Josa': 13,
 '다/Adverb': 14,
 '다그/Noun': 15,
 '더빙/Noun': 16,
 '도/Josa': 17,
 '떼다/Verb': 18,
 '래서/Noun': 19,
 '마/Noun': 20,
 '막/Noun': 21,
 '목소리/Noun': 22,
 '무재/Noun': 23,
 '밓었/Noun': 24,
 '반개/Noun': 25,
 '별/Modifier': 26,
 '보다/Verb': 27,
 '부터/Josa': 28,
 '살다/Verb': 29,
 '살리다/Verb': 30,
 '생인/Noun': 31,
 '세/Noun': 32,
 '솔직하다/Adjective': 33,
 '아/Exclamation': 34,
 '아깝다/Adjective': 35,
 '없다/Adjective': 36,
 '영화/Noun': 37,
 '움/Noun': 38,
 '원작/Noun': 39,
 '을/Josa': 40,
 '의/Josa': 41,
 '이야기/Noun': 42,
 '재미/Noun': 43,
 '제대로/Noun': 44,
 '조정/Noun': 45,
 '진짜/Noun': 46,
 '짜증나다/Adjective': 47,
 '초등학교/Noun': 48,
 '추천/Noun': 49,
 '평점/Noun': 50,
 '하다/Verb': 51,
 '학년/Noun': 52,
 '한/Josa': 53}

In [488]:
idx_tk

{1: '.../Punctuation',
 2: '../Punctuation',
 3: './Punctuation',
 4: '1/Number',
 5: '3/Number',
 6: '8/Number',
 7: 'ㅋㅋㅋ/KoreanParticle',
 8: '걸음/Noun',
 9: '교도소/Noun',
 10: '구먼/Noun',
 11: '긴장감/Noun',
 12: '너/Modifier',
 13: '는/Josa',
 14: '다/Adverb',
 15: '다그/Noun',
 16: '더빙/Noun',
 17: '도/Josa',
 18: '떼다/Verb',
 19: '래서/Noun',
 20: '마/Noun',
 21: '막/Noun',
 22: '목소리/Noun',
 23: '무재/Noun',
 24: '밓었/Noun',
 25: '반개/Noun',
 26: '별/Modifier',
 27: '보다/Verb',
 28: '부터/Josa',
 29: '살다/Verb',
 30: '살리다/Verb',
 31: '생인/Noun',
 32: '세/Noun',
 33: '솔직하다/Adjective',
 34: '아/Exclamation',
 35: '아깝다/Adjective',
 36: '없다/Adjective',
 37: '영화/Noun',
 38: '움/Noun',
 39: '원작/Noun',
 40: '을/Josa',
 41: '의/Josa',
 42: '이야기/Noun',
 43: '재미/Noun',
 44: '제대로/Noun',
 45: '조정/Noun',
 46: '진짜/Noun',
 47: '짜증나다/Adjective',
 48: '초등학교/Noun',
 49: '추천/Noun',
 50: '평점/Noun',
 51: '하다/Verb',
 52: '학년/Noun',
 53: '한/Josa'}

In [490]:
vocab_size=len(tk_idx)+1
vocab_size

54

In [492]:
tk_ck_data

[['아/Exclamation',
  '더빙/Noun',
  '../Punctuation',
  '진짜/Noun',
  '짜증나다/Adjective',
  '목소리/Noun'],
 ['너/Modifier',
  '무재/Noun',
  '밓었/Noun',
  '다그/Noun',
  '래서/Noun',
  '보다/Verb',
  '추천/Noun',
  '한/Josa',
  '다/Adverb'],
 ['교도소/Noun',
  '이야기/Noun',
  '구먼/Noun',
  '../Punctuation',
  '솔직하다/Adjective',
  '재미/Noun',
  '는/Josa',
  '없다/Adjective',
  '../Punctuation',
  '평점/Noun',
  '조정/Noun'],
 ['막/Noun',
  '걸음/Noun',
  '마/Noun',
  '떼다/Verb',
  '3/Number',
  '세/Noun',
  '부터/Josa',
  '초등학교/Noun',
  '1/Number',
  '학년/Noun',
  '생인/Noun',
  '8/Number',
  '살다/Verb',
  '영화/Noun',
  './Punctuation',
  'ㅋㅋㅋ/KoreanParticle',
  '.../Punctuation',
  '별/Modifier',
  '반개/Noun',
  '도/Josa',
  '아깝다/Adjective',
  '움/Noun',
  './Punctuation'],
 ['원작/Noun',
  '의/Josa',
  '긴장감/Noun',
  '을/Josa',
  '제대로/Noun',
  '살리다/Verb',
  '하다/Verb',
  './Punctuation']]

In [512]:
X=[]
y=[]
ts=5
for s in tk_ck_data:
    for i in range(ts,len(s)):
        input_s=s[i-ts:i]
        t=s[i]
        if all(tk_w in tk_idx for tk_w in input_s+[t]):
            X.append([tk_idx[tk_w] for tk_w in input_s])
            y.append(tk_idx[t])
X=np.array(X)
y=np.array(y)

In [514]:
vocab_size

54

In [518]:
from keras.losses import categorical_crossentropy, sparse_categorical_crossentropy

array([22, 27, 49, 53, 14, 43, 13, 36,  2, 50, 45, 32, 28, 48,  4, 52, 31,
        6, 29, 37,  3,  7,  1, 26, 25, 17, 35, 38,  3, 30, 51,  3])

In [522]:
m=Sequential()
m.add(Embedding(vocab_size,30))
m.add(LSTM(100))
m.add(Dense(vocab_size,activation='softmax'))
m.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['acc'])
m.fit(X,y,epochs=200)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - acc: 0.0312 - loss: 3.9899
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - acc: 0.0312 - loss: 3.9863
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - acc: 0.0938 - loss: 3.9826
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - acc: 0.1250 - loss: 3.9789
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - acc: 0.1562 - loss: 3.9750
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - acc: 0.2500 - loss: 3.9710
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - acc: 0.2500 - loss: 3.9667
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - acc: 0.2812 - loss: 3.9623
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - acc: 0.2812 - loss: 3.9575
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - acc: 0.2500 - loss: 3.9523
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - acc: 0.2500 - loss: 3.9468
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - acc: 0.2188 - loss: 3.9408
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step 

In [534]:
idx_tk[m.predict(X[:1]).argmax()].split('/')[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


'목소리'

In [550]:
ck_l_data=list(X[0][1:])
ck_l_data.append(m.predict(X[:1]).argmax())
ck_l_data

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


[16, 2, 46, 47, 22]

In [552]:
#문장 생성의 모델 구조는 일반적인 생성 모델 구조를 갖는다 (단순 신경망을 이용)->(LLM)
#출력 결과를 도출하는 후처리기의 역량이 문장 생성의 모델의 핵심이다.-> (프롬프트(입력과 출력))

In [ ]:
data2=pd.read_table('steam.txt',names=['y','x'])
